# Assignment 1: Custom Missing Value Imputer
**Course:** Feature Engineering & MLOps (Unit 1, Session 4)  
**Student Name:** Manas More  
**Assignment No:** 1  
**Topic:** Object-Oriented Imputation Class following Scikit-Learn API  

---

### Overview & Objectives
In this assignment, I designed and built a modular scikit-learn compatible transformer called `CustomImputer`. Rather than relying on generic black-box tools, implementing this transformer from first principles illustrates the core mechanics of missing data imputation and safe machine learning pipelines.

Key requirements addressed in this notebook:
1. **Scikit-Learn API Standards:** Subclass `BaseEstimator` and `TransformerMixin`, implement `.fit()` and `.transform()`, ensuring `.fit()` returns `self` and `.transform()` yields a clean copy.
2. **Strict Train-Only Fitting:** Calculate all fill statistics (median, mean, mode) exclusively from the training split to avoid data leakage.
3. **Automatic Dtype Detection:** Automatically distinguish numeric predictors from categorical features using pandas data types without needing hardcoded column lists.
4. **Missingness Indicator Generation:** Generate binary indicator columns (`<feature>_was_missing`) to preserve the valuable predictive signal that a value was originally unobserved (vital for MNAR / MAR mechanisms).
5. **State Validation & Guard Rails:** Raise a descriptive `NotFittedError` if `.transform()` is called before `.fit()`.
6. **Bonus (Per-Column Overrides):** Provide a `column_overrides` parameter to customize imputation strategies for individual features independently of global defaults.

## 1. Imports and Environment Setup

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Global display configuration and seed setup
pd.set_option('display.max_columns', None)
np.random.seed(42)
print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Implementing the `CustomImputer` Class

Below is the complete implementation of `CustomImputer`:
- **In `fit()`:** We detect numerical vs. categorical datatypes, calculate fill statistics solely on the training partition, and store them in `self.learned_fill_values_`.
- **In `transform()`:** We ensure the imputer was fitted using `check_is_fitted()`, append missingness indicator flags for columns that had nulls in training, and impute missing cells using the saved statistics.

In [2]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """
    Custom scikit-learn transformer for mixed tabular missing data imputation.
    
    Parameters:
    -----------
    numeric_strategy : str, default='median'
        Default strategy for numerical columns ('median' or 'mean').
    categorical_strategy : str, default='most_frequent'
        Default strategy for non-numerical columns ('most_frequent').
    add_missing_indicator : bool, default=True
        Whether to create binary indicator columns for features with nulls in training data.
    column_overrides : dict or None, default=None
        Optional mapping of column names to specific strategies,
        e.g. {'weekly_study_hours': 'mean', 'income_bracket': 'most_frequent'}.
    """
    def __init__(self, numeric_strategy='median', categorical_strategy='most_frequent',
                 add_missing_indicator=True, column_overrides=None):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _determine_column_strategy(self, col_name, is_numeric):
        """Determine the effective imputation strategy for a target feature."""
        if self.column_overrides and col_name in self.column_overrides:
            return self.column_overrides[col_name]
        return self.numeric_strategy if is_numeric else self.categorical_strategy

    def fit(self, X, y=None):
        """
        Compute replacement values for each feature from the training split X.
        """
        # Ensure input is a DataFrame copy
        df_in = pd.DataFrame(X).copy()
        
        self.feature_names_in_ = list(df_in.columns)
        
        # Automatically classify numeric vs categorical columns
        self.numeric_columns_ = [
            col for col in df_in.columns 
            if pd.api.types.is_numeric_dtype(df_in[col])
        ]
        self.categorical_columns_ = [
            col for col in df_in.columns 
            if col not in self.numeric_columns_
        ]
        
        # Identify columns exhibiting null entries in training
        self.columns_with_missing_ = [
            col for col in df_in.columns if df_in[col].isnull().any()
        ]
        
        self.learned_fill_values_ = {}
        for col in df_in.columns:
            is_num = col in self.numeric_columns_
            strategy = self._determine_column_strategy(col, is_num)
            valid_vals = df_in[col].dropna()
            
            if is_num:
                if strategy == 'median':
                    self.learned_fill_values_[col] = valid_vals.median()
                elif strategy == 'mean':
                    self.learned_fill_values_[col] = valid_vals.mean()
                else:
                    raise ValueError(f"Unknown numeric strategy '{strategy}' for column '{col}'")
            else:
                if strategy == 'most_frequent':
                    modes = valid_vals.mode()
                    self.learned_fill_values_[col] = modes.iloc[0] if len(modes) > 0 else np.nan
                else:
                    raise ValueError(f"Unknown categorical strategy '{strategy}' for column '{col}'")
                    
        return self

    def transform(self, X):
        """
        Impute missing values using learned statistics and append missing indicator columns if enabled.
        """
        # Validation check: verify fit was executed
        check_is_fitted(self, 'learned_fill_values_')
        
        df_out = pd.DataFrame(X).copy()
        
        # Step 1: Create missingness indicator flags for columns that had nulls in training
        if self.add_missing_indicator:
            for col in self.columns_with_missing_:
                if col in df_out.columns:
                    df_out[f"{col}_was_missing"] = df_out[col].isnull().astype(int)
        
        # Step 2: Impute nulls using previously learned values without recomputing
        for col in df_out.columns:
            if col in self.learned_fill_values_ and df_out[col].isnull().any():
                df_out[col] = df_out[col].fillna(self.learned_fill_values_[col])
                
        return df_out


## 3. Dataset Ingestion and Missing Value Analysis

We load the `student_performance_raw.csv` dataset and review the columns containing missing entries.

In [3]:
# Support relative path whether executing from repo root or inside Assignments folder
data_file = '../data/raw/student_performance_raw.csv'
if not os.path.exists(data_file):
    data_file = 'data/raw/student_performance_raw.csv'

raw_df = pd.read_csv(data_file)
print(f"Dataset shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")

# Summary of missing values
missing_counts = raw_df.isnull().sum()
cols_with_nulls = missing_counts[missing_counts > 0]
missing_overview = pd.DataFrame({
    'Null Count': cols_with_nulls,
    'Percentage': (cols_with_nulls / len(raw_df) * 100).round(2)
})
missing_overview

Dataset shape: 600 rows, 17 columns


,Null Count,Percentage
weekly_study_hours,36,6.00
income_bracket,30,5.00
prev_exam_score,25,4.17
mock_test_3,21,3.50
feedback_text,68,11.33


### Examination of Missing Data Mechanisms:
As covered in our coursework, this dataset contains instances of all three key missingness mechanisms:
- **MCAR (Missing Completely at Random):** `weekly_study_hours` is missing at random, with no systematic relation to study patterns or student grades.
- **MAR (Missing at Random):** `prev_exam_score` missingness depends conditionally on observable features (`batch_type` / enrollment period) rather than the missing score itself.
- **MNAR (Missing Not at Random):** `mock_test_3` missingness is directly related to the missing value itself (students struggling academically were systematically more likely to skip mock test 3).

## 4. Train / Test Split (Strict 80/20 Separation)

We drop `student_id` because it is an arbitrary record identifier with no predictive power, and set `final_score` as our regression target. We perform the train/test split **prior to** imputation to prevent any data leakage between evaluation and training partitions.

In [4]:
predictor_cols = [c for c in raw_df.columns if c not in ('student_id', 'final_score')]

X = raw_df[predictor_cols].copy()
y = raw_df['final_score'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (480, 15)
X_test shape:  (120, 15)


## 5. Fitting and Applying `CustomImputer`

We fit `CustomImputer` strictly on `X_train` with `numeric_strategy='median'`, `categorical_strategy='most_frequent'`, and `add_missing_indicator=True`.

In [5]:
# Instantiate imputer with default settings
imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    add_missing_indicator=True
)

# Fit strictly on training split
imputer.fit(X_train)

# Transform both splits
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Detected numeric columns:", imputer.numeric_columns_)
print("\nDetected categorical columns:", imputer.categorical_columns_)
print("\nColumns with missing values in train set:", imputer.columns_with_missing_)
print("\nLearned fill values:")
for col, val in imputer.learned_fill_values_.items():
    print(f"  {col}: {val}")

print(f"\nNew shape of X_train after indicators: {X_train_imputed.shape}")
print(f"New shape of X_test after indicators:  {X_test_imputed.shape}")

Detected numeric columns: ['city_tier', 'age', 'attendance_pct', 'weekly_study_hours', 'prev_exam_score', 'mock_test_1', 'mock_test_2', 'mock_test_3', 'doubt_sessions_attended']

Detected categorical columns: ['city', 'course', 'batch_type', 'enrollment_date', 'income_bracket', 'feedback_text']

Columns with missing values in train set: ['weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_3', 'feedback_text']

Learned fill values:
  city: Mumbai
  city_tier: 1.0
  course: NEET
  batch_type: Weekday
  age: 17.0
  enrollment_date: 2024-10-23
  attendance_pct: 78.35
  weekly_study_hours: 5.0
  income_bracket: 5-10L
  prev_exam_score: 66.1
  mock_test_1: 66.55
  mock_test_2: 66.35
  mock_test_3: 71.3
  doubt_sessions_attended: 4.0
  feedback_text: Need more practice sheets for weak topics

New shape of X_train after indicators: (480, 20)
New shape of X_test after indicators:  (120, 20)


## 6. Verification and Guard Rail Validation

We confirm that:
1. All null entries have been resolved across both train and test splits.
2. Calling `transform()` on an unfitted instance appropriately raises a `NotFittedError`.

In [6]:
# Check null counts in the original feature columns
train_nulls_left = X_train_imputed[imputer.feature_names_in_].isnull().sum().sum()
test_nulls_left = X_test_imputed[imputer.feature_names_in_].isnull().sum().sum()

print(f"Null values left in X_train: {train_nulls_left}")
print(f"Null values left in X_test:  {test_nulls_left}")

assert train_nulls_left == 0, "Assertion Error: Nulls still present in X_train!"
assert test_nulls_left == 0, "Assertion Error: Nulls still present in X_test!"
print("PASSED: 0 missing values remain in both train and test splits.")

Null values left in X_train: 0
Null values left in X_test:  0
PASSED: 0 missing values remain in both train and test splits.


In [7]:
# Guard rail check: calling transform on an unfitted imputer
test_imputer = CustomImputer()
try:
    test_imputer.transform(X_train)
    print("FAILED: Guard rail did not raise error.")
except NotFittedError as err:
    print("PASSED: Guard rail caught unfitted transform as expected.")
    print("Error message:", err)

PASSED: Guard rail caught unfitted transform as expected.
Error message: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## 7. Distribution Analysis: Mean and Variance Before vs. After Imputation

We analyze how imputing missing values influences the central tendency (mean) and spread (standard deviation) of continuous predictors.

In [8]:
num_cols_missing = [c for c in imputer.numeric_columns_ if c in imputer.columns_with_missing_]

stats_rows = []
for col in num_cols_missing:
    raw_col = X_train[col]
    clean_col = X_train_imputed[col]
    stats_rows.append({
        'Feature': col,
        'Mean (Before)': raw_col.mean(),
        'Mean (After)': clean_col.mean(),
        'Mean Shift': clean_col.mean() - raw_col.mean(),
        'Std (Before)': raw_col.std(),
        'Std (After)': clean_col.std(),
        'Std Shift': clean_col.std() - raw_col.std(),
    })

stats_comparison_df = pd.DataFrame(stats_rows).round(4)
stats_comparison_df

,Feature,Mean (Before),Mean (After),Mean Shift,Std (Before),Std (After),Std Shift
0,weekly_study_hours,6.0642,6.0088,-0.0554,4.3503,4.2418,-0.1084
1,prev_exam_score,65.8245,65.8377,0.0132,14.3708,14.0217,-0.3491
2,mock_test_3,70.7015,70.7202,0.0187,19.0773,18.7765,-0.3008


### Discussion of Summary Statistics:
- **Central Tendency (Mean):** The mean shifts only marginally across features (e.g., `weekly_study_hours` shifts from 6.064 to 6.009). Because we used median imputation, the center of distribution remains resilient to outliers.
- **Spread (Standard Deviation):** Standard deviation consistently decreases across all imputed numeric columns (e.g., `prev_exam_score` std drops from 14.37 to 14.02). Replacing missing entries with an identical constant value contributes zero variance, pulling the overall dispersion inward toward the central estimate.

## 8. Validation Against Scikit-Learn's `SimpleImputer`

To cross-validate our custom logic, we fit scikit-learn's standard `SimpleImputer(strategy='median')` on the identical training data and confirm that our computed fill values match exactly.

In [9]:
sklearn_imputer = SimpleImputer(strategy='median')
sklearn_imputer.fit(X_train[num_cols_missing])

validation_table = pd.DataFrame({
    'Feature': num_cols_missing,
    'CustomImputer Value': [imputer.learned_fill_values_[c] for c in num_cols_missing],
    'SimpleImputer Value': sklearn_imputer.statistics_,
})
validation_table['Matches Exact'] = np.isclose(
    validation_table['CustomImputer Value'], validation_table['SimpleImputer Value']
)
print(validation_table)

assert validation_table['Matches Exact'].all(), "Sanity check failed: values do not match!"
print("\nPASSED: CustomImputer fill values match SimpleImputer exactly.")

              Feature  CustomImputer Value  SimpleImputer Value  Matches Exact
0  weekly_study_hours                  5.0                  5.0           True
1     prev_exam_score                 66.1                 66.1           True
2         mock_test_3                 71.3                 71.3           True

PASSED: CustomImputer fill values match SimpleImputer exactly.


## 9. Reflection Questions

**1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?**  
Calling `fit()` across the full dataset or on the test partition leaks information (such as target distributions, means, medians, or category frequencies) from unseen evaluation data into the model training phase. This produces unrealistically optimistic performance metrics and conceals how the model will generalize to real-world data. In production, future records are completely unavailable during training, so fitting solely on `X_train` enforces realistic deployment conditions.

**2. `mock_test_3` is MNAR (Session 4). Does mean/median imputation genuinely solve the problem for this column? What does your `add_missing_indicator` feature contribute that plain imputation does not?**  
No, standard median imputation does not genuinely solve the underlying problem for an MNAR column like `mock_test_3`. In this scenario, students who skipped or opted out of mock test 3 were systematically more likely to be struggling academically; replacing their missing scores with the cohort median (~71.3) artificially portrays lower-performing students as average performers. The `add_missing_indicator` column (`mock_test_3_was_missing`) solves this by providing a distinct binary feature indicating that the exam was omitted, allowing downstream algorithms to learn that the omission itself carries a strong negative signal regarding final academic performance.

**3. Suppose a brand-new column, entirely missing in the training data but present in the test data, is passed to your imputer. What does your current implementation do — and what SHOULD a production-grade version do instead?**  
In our current implementation, because the new feature was not present during `fit()`, it is not recorded in `self.learned_fill_values_`. During `transform()`, the check `if col in self.learned_fill_values_` evaluates to `False`, allowing the feature to pass through untouched with all of its missing values intact. In production, this silent pass-through could cause downstream estimators to fail due to unhandled nulls. A production-grade imputer should perform strict schema validation against `feature_names_in_`, raising a descriptive error or handling unexpected columns according to a defined fallback policy (such as dropping unauthorized columns or filling with a predetermined constant).

## 10. Bonus (+10% Extra Credit): Feature-Specific Strategy Overrides

We verify the `column_overrides` capability by applying a `mean` strategy specifically to `weekly_study_hours` while retaining the default `median` strategy across all other numerical features.

In [10]:
override_imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    column_overrides={
        'weekly_study_hours': 'mean',       # Overridden to mean
        'income_bracket': 'most_frequent'  # Explicit categorical override
    }
)

override_imputer.fit(X_train)

default_study_val = imputer.learned_fill_values_['weekly_study_hours']
override_study_val = override_imputer.learned_fill_values_['weekly_study_hours']

print(f"Default numeric fill value for weekly_study_hours (median): {default_study_val:.4f}")
print(f"Overridden fill value for weekly_study_hours (mean):        {override_study_val:.4f}")
print(f"income_bracket fill value:                                 {override_imputer.learned_fill_values_['income_bracket']}")

assert default_study_val != override_study_val, "Override failed to change fill value!"
print("\nPASSED: Column override correctly modified weekly_study_hours strategy without altering other columns.")

Default numeric fill value for weekly_study_hours (median): 5.0000
Overridden fill value for weekly_study_hours (mean):        6.0642
income_bracket fill value:                                 5-10L

PASSED: Column override correctly modified weekly_study_hours strategy without altering other columns.
